# Numerical approximation of extrinsic curvature

In this notebook we test the convergence of the lifted generalized Weingarten tensor for several geometries. The geometries are generated by plain functions so the construction is visible: first the reference domain, then the embedding, then the exact normal and Weingarten tensor used for the error computation.


<style>
.curv-grid { display: grid; grid-template-columns: repeat(auto-fit, minmax(250px, 1fr)); gap: 1rem; align-items: start; }
.curv-fig { margin: 0; }
.curv-fig img { width: 100%; border-radius: 6px; }
.curv-fig figcaption { font-size: 0.9rem; color: #555; margin-top: 0.35rem; }
.curv-callout { border-left: 4px solid #386cb0; padding: 0.65rem 0.9rem; background: #f6f8fb; margin: 1rem 0; }
.curv-eqbox { border: 1px solid #d6dbe6; border-radius: 6px; padding: 0.8rem 1rem; background: #fbfcff; }
</style>


In [ ]:

import matplotlib.pyplot as plt
from netgen.occ import OCCGeometry, Circle, MoveTo, X, Y, Rectangle, IdentificationType, gp_Trsf
from ngsolve import *
from ngsolve.krylovspace import CGSolver
from ngsolve.webgui import Draw

## Curvature, lifting, and convergence helpers

In [ ]:
def exact_weingarten_from_normal(normal, coords=(x, y, z), transpose=True):
    P = Id(3) - OuterProduct(normal, normal)
    Dnormal = CF(tuple(normal.Diff(coord) for coord in coords), dims=(3, 3))
    if transpose:
        Dnormal = Dnormal.trans
    return P * Dnormal * P


def compute_lifted_shape_operator(mesh, order, dirichlet=".*", K_ex=None, neumann="", nv_ex=None):
    if order < 1:
        raise ValueError("Order must be at least 1")

    n = specialcf.normal(3)
    t = specialcf.tangential(3)
    mu = Cross(n, t)

    ind_neumann = GridFunction(FacetSurface(mesh, order=0))
    ind_neumann.Set(1, definedon=mesh.BBoundaries(neumann))

    gf_average_nv = GridFunction(VectorFacetSurface(mesh, order=order - 1))
    gf_average_nv.Set(n, dual=True, definedon=mesh.Boundaries(".*"))
    cf_average_nv = Normalize(gf_average_nv)

    fes = Periodic(HDivDivSurface(mesh, order=order - 1, dirichlet_bbnd=dirichlet))
    sigma, tau = fes.TnT()
    sigma, tau = sigma.Trace(), tau.Trace()

    mass = BilinearForm(
        InnerProduct(sigma, tau) * ds,
        symmetric=True,
        symmetric_storage=True,
        condense=order > 1,
    )

    rhs = LinearForm(fes)
    rhs += InnerProduct(Grad(n), tau) * ds
    rhs += (1 - ind_neumann) * (pi / 2 - acos(cf_average_nv * mu)) * tau[mu, mu] * ds(element_boundary=True)

    if nv_ex:
        rhs += -ind_neumann * (n - nv_ex) * mu * tau[mu, mu] * ds(element_boundary=True)

    gf_kappa = GridFunction(fes)
    if K_ex:
        gf_kappa.Set(K_ex, definedon=mesh.Boundaries(".*"), dual=True)

    pre = Preconditioner(mass, "bddc")
    mass.Assemble()
    rhs.Assemble()

    residual = gf_kappa.vec.CreateVector()
    mass.Apply(gf_kappa.vec, residual)
    residual.data -= rhs.vec

    if mass.condense:
        invS = CGSolver(mass.mat, pre.mat, printing=False, maxiter=400)
        ext = IdentityMatrix() + mass.harmonic_extension
        inv = mass.inner_solve + ext @ invS @ ext.T
    else:
        inv = mass.mat.Inverse(fes.FreeDofs(), inverse="sparsecholesky")

    gf_kappa.vec.data -= inv * residual
    return gf_kappa


def compute_hm1_norm(func_vol, func_bnd, ex_solution, mesh, order):
    fes = H1(mesh, order=order, dirichlet_bbnd=".*")
    u, v = fes.TnT()

    form = BilinearForm(fes, symmetric=True, symmetric_storage=True, condense=True)
    form += (u * v + Grad(u).Trace() * Grad(v).Trace()) * ds

    pre = Preconditioner(form, "bddc")
    form.Assemble()

    invS = CGSolver(form.mat, pre, printing=False, maxiter=600)
    if form.condense:
        ext = IdentityMatrix() + form.harmonic_extension
        inv = form.inner_solve + ext @ invS @ ext.T
    else:
        inv = invS

    rhs = MultiVector(fes.ndof, 6, False)
    gf_lift = GridFunction(fes**6)

    for i in range(6):
        j = i if i < 3 else (i + 1 if i < 5 else 8)
        linear = LinearForm(
            InnerProduct(func_vol[j] - ex_solution[j], v) * ds
            + InnerProduct(func_bnd[j], v) * ds(element_boundary=True)
        ).Assemble()
        rhs[i].data = linear.vec

    lift_vecs = (inv * rhs).Evaluate()
    for i in range(6):
        gf_lift.components[i].vec.data = lift_vecs[i]

    index_sym_matrix = [0, 1, 2, 1, 3, 4, 2, 4, 5]
    lifted = CF(tuple(gf_lift.components[i] for i in index_sym_matrix), dims=(3, 3))
    grad_lifted = CF(tuple(Grad(gf_lift.components[i]) for i in index_sym_matrix), dims=(3, 3, 3))

    return sqrt(Integrate(InnerProduct(lifted, lifted) + InnerProduct(grad_lifted, grad_lifted), mesh, BND))


def compute_hm1_error(mesh, order, K_ex, K_lifted=None):
    if order < 1:
        raise ValueError("Order must be at least 1")

    n = specialcf.normal(3)
    t = specialcf.tangential(3)
    mu = Cross(n, t)

    gf_average_nv = GridFunction(VectorFacetSurface(mesh, order=order - 1))
    gf_average_nv.Set(n, dual=True, definedon=mesh.Boundaries(".*"))
    cf_average_nv = Normalize(gf_average_nv)

    if K_lifted:
        vol_term = K_lifted
        bnd_term = CF(0) * OuterProduct(mu, mu)
    else:
        vol_term = Grad(n)
        bnd_term = (pi / 2 - acos(cf_average_nv * mu)) * OuterProduct(mu, mu)

    return compute_hm1_norm(vol_term, bnd_term, K_ex, mesh, order=order + 2)


def compute_lifted_curvature_convergence(example, order=3, canonical_interp=False, maxhs=None, draw_last=False):
    if maxhs is None:
        maxhs = [2 ** (-i) for i in range(5)]

    l2err = []
    hm1err = []
    hm1liftederr = []
    l2rate = []
    hm1rate = []
    hm1liftedrate = []
    ndof = []
    last = None

    with TaskManager():
        for maxh in maxhs:
            print(f"{example['name']}: maxh={maxh:.4f}")
            mesh = Mesh(example["geometry"].GenerateMesh(maxh=maxh))

            gf_Phi = GridFunction(VectorH1(mesh, order=order))
            gf_Phi.Set(
                example["embedding"] - CF((x, y, 0)),
                definedon=mesh.Boundaries(".*"),
                dual=canonical_interp,
            )

            mesh.SetDeformation(gf_Phi)
            K_lifted = compute_lifted_shape_operator(
                mesh=mesh,
                order=order,
                dirichlet=example["boundaries"][0],
                K_ex=example["Weingarten"],
                neumann="|".join(example["boundaries"][1:]) if len(example["boundaries"]) > 1 else "",
                nv_ex=example["normal"],
            )

            l2err.append(
                sqrt(
                    Integrate(
                        InnerProduct(K_lifted - example["Weingarten"], K_lifted - example["Weingarten"]),
                        mesh,
                        BND,
                    )
                )
            )
            hm1err.append(compute_hm1_error(mesh=mesh, order=order, K_ex=example["Weingarten"]))
            hm1liftederr.append(
                compute_hm1_error(mesh=mesh, order=order, K_ex=example["Weingarten"], K_lifted=K_lifted)
            )

            if len(l2err) > 1:
                l2rate.append(log(l2err[-2] / l2err[-1]) / log(maxhs[len(l2err) - 2] / maxhs[len(l2err) - 1]))
                hm1rate.append(log(hm1err[-2] / hm1err[-1]) / log(maxhs[len(hm1err) - 2] / maxhs[len(hm1err) - 1]))
                hm1liftedrate.append(
                    log(hm1liftederr[-2] / hm1liftederr[-1])
                    / log(maxhs[len(hm1liftederr) - 2] / maxhs[len(hm1liftederr) - 1])
                )

            ndof.append(K_lifted.space.ndof)
            last = (mesh, K_lifted)
            mesh.UnsetDeformation()

    result = {
        "name": example["name"],
        "order": order,
        "maxhs": maxhs,
        "l2err": [float(v) for v in l2err],
        "hm1err": [float(v) for v in hm1err],
        "hm1liftederr": [float(v) for v in hm1liftederr],
        "l2rate": [float(v) for v in l2rate],
        "hm1rate": [float(v) for v in hm1rate],
        "hm1liftedrate": [float(v) for v in hm1liftedrate],
        "ndof": ndof,
    }

    if draw_last and last is not None:
        mesh, K_lifted = last
        Draw(0.5 * Trace(K_lifted), mesh, f"mean curvature, {example['name']}", deformation=gf_Phi)

    return result


def plot_convergence(result):
    ndof = result["ndof"]
    order = result["order"]
    fig, ax = plt.subplots(figsize=(5.5, 4))
    ax.loglog(ndof, result["l2err"], "o-", label="$L^2$")
    ax.loglog(ndof, result["hm1err"], "s-", label="$H^{-1}$")
    ax.loglog(ndof, result["hm1liftederr"], "^-", label="lifted $H^{-1}$")

    if ndof and result["hm1err"]:
        ref_k = [result["hm1err"][0] * (ndof[0] / n) ** (order / 2) for n in ndof]
        ref_kp1 = [result["hm1err"][0] * (ndof[0] / n) ** ((order + 1) / 2) for n in ndof]
        ax.loglog(ndof, ref_k, "k--", lw=1.2, label=rf"$N^{{-{order}/2}}$")
        ax.loglog(ndof, ref_kp1, "k:", lw=1.5, label=rf"$N^{{-({order}+1)/2}}$")

    ax.grid(True, which="both", ls=":", lw=0.8)
    ax.set_xlabel("number of dofs $N$")
    ax.set_ylabel("error")
    ax.set_title(result["name"])
    ax.legend()
    plt.show()


def print_result(result):
    print(result["name"])
    print("  L2 errors:     ", result["l2err"])
    print("  L2 rates:      ", result["l2rate"])
    print("  H-1 errors:    ", result["hm1err"])
    print("  H-1 rates:     ", result["hm1rate"])
    print("  lifted H-1 errors:", result["hm1liftederr"])
    print("  lifted H-1 rates: ", result["hm1liftedrate"])
    print("  Number of dofs:", result["ndof"])
    plot_convergence(result)


## Geometry generators

Each function returns a dictionary containing the reference-domain geometry, the embedding into $\mathbb R^3$, the exact normal, the exact Weingarten tensor, and the boundary names used by the lifting routine.


In [ ]:
def make_half_sphere(radius=1.3):
    R = radius
    r_ref = sqrt(x**2 + y**2)
    phi = atan2(y, x)
    embedding = R * CF((cos(phi) * sin(r_ref * pi / 2), sin(phi) * sin(r_ref * pi / 2), cos(r_ref * pi / 2)))

    radius_ex = sqrt(x**2 + y**2 + z**2)
    x_ex = R * x / radius_ex
    y_ex = R * y / radius_ex
    z_ex = R * z / radius_ex
    surface = x_ex**2 + y_ex**2 + z_ex**2 - R**2
    normal = Normalize(CF((surface.Diff(x_ex), surface.Diff(y_ex), surface.Diff(z_ex))))
    Weingarten = exact_weingarten_from_normal(normal, coords=(x_ex, y_ex, z_ex), transpose=True)

    face = Circle((0, 0), R).Face()
    face.edges.name = "bnd"
    return {"name": f"Half sphere, R={R}", "geometry": OCCGeometry(face), "embedding": embedding, "normal": normal, "Weingarten": Weingarten, "boundaries": ["bnd"]}


def make_ellipsoid_part(a=1, b=1.5, c=0.8):
    embedding = CF((a * cos(x) * sin(y), b * sin(x) * sin(y), c * cos(y)))

    radius_ex = sqrt((x / a) ** 2 + (y / b) ** 2 + (z / c) ** 2)
    x_ex = x / radius_ex
    y_ex = y / radius_ex
    z_ex = z / radius_ex
    surface = x_ex**2 / a**2 + y_ex**2 / b**2 + z_ex**2 / c**2 - 1
    normal = Normalize(CF((surface.Diff(x_ex), surface.Diff(y_ex), surface.Diff(z_ex))))
    Weingarten = exact_weingarten_from_normal(normal, coords=(x_ex, y_ex, z_ex), transpose=True)

    face = MoveTo(pi / 2, -pi / 2).Rectangle(pi / 2, pi / 2 - pi / 6).Face()
    face.edges.Min(X).name = "left"
    face.edges.Max(X).name = "right"
    face.edges.Min(Y).name = "top"
    face.edges.Max(Y).name = "bottom"
    return {"name": f"Ellipsoid part, a={a}, b={b}, c={c}", "geometry": OCCGeometry(face), "embedding": embedding, "normal": normal, "Weingarten": Weingarten, "boundaries": ["left", "right", "top", "bottom"]}


def make_hyperboloid(R=1.5):
    embedding = CF((R * cos(pi * x - pi / 2) * cosh(2 * y - 1), R * sinh(2 * y - 1), R * sin(-pi * x + pi / 2) * cosh(2 * y - 1)))

    radius_ex = sqrt(x**2 / R**2 - y**2 / R**2 + z**2 / R**2)
    x_ex = x / radius_ex
    y_ex = y / radius_ex
    z_ex = z / radius_ex
    surface = x_ex**2 / R**2 - y_ex**2 / R**2 + z_ex**2 / R**2 - 1
    normal = Normalize(CF((surface.Diff(x_ex), surface.Diff(y_ex), surface.Diff(z_ex))))
    Weingarten = exact_weingarten_from_normal(normal, coords=(x_ex, y_ex, z_ex), transpose=True)

    face = Rectangle(1, 1).Face()
    face.edges.Min(X).name = "left"
    face.edges.Max(X).name = "right"
    face.edges.Min(Y).name = "bottom"
    face.edges.Max(Y).name = "top"
    return {"name": f"Hyperboloid, R={R}", "geometry": OCCGeometry(face), "embedding": embedding, "normal": normal, "Weingarten": Weingarten, "boundaries": ["left", "right", "bottom", "top"]}


def make_pseudosphere():
    def gu(u):
        return log(tan(u / 2)) + cos(u)

    def acosh_cf(v):
        return log(sqrt(1 / v - 1) * sqrt(1 / v + 1) + 1 / v)

    u = pi / 9 - x * pi / 18
    embedding = CF((sin(u) * cos(y * 2 * pi), sin(u) * sin(y * 2 * pi), gu(u)))

    surface = (acosh_cf(sqrt(x**2 + y**2)) - sqrt(1 - x**2 - y**2)) ** 2 - z**2
    normal = -Normalize(CF((surface.Diff(x), surface.Diff(y), surface.Diff(z))))
    Weingarten = exact_weingarten_from_normal(normal, transpose=False)

    face = Rectangle(1, 1).Face()
    face.edges.Min(X).name = "left"
    face.edges.Max(X).name = "right"
    face.edges.Min(Y).name = "bottom"
    face.edges.Max(Y).name = "top"
    return {"name": "Pseudosphere", "geometry": OCCGeometry(face), "embedding": embedding, "normal": normal, "Weingarten": Weingarten, "boundaries": ["left", "right", "bottom", "top"]}


def make_catenoid():
    embedding = CF((cosh(2 * (x - 0.5)) * cos(y * 2 * pi), cosh(2 * (x - 0.5)) * sin(y * 2 * pi), 2 * (x - 0.5)))

    surface = x**2 + y**2 - cosh(z) ** 2
    normal = -Normalize(CF((surface.Diff(x), surface.Diff(y), surface.Diff(z))))
    Weingarten = exact_weingarten_from_normal(normal, transpose=False)

    face = Rectangle(1, 1).Face()
    face.edges.Min(X).name = "left"
    face.edges.Max(X).name = "right"
    face.edges.Min(Y).name = "bottom"
    face.edges.Max(Y).name = "top"
    return {"name": "Catenoid", "geometry": OCCGeometry(face), "embedding": embedding, "normal": normal, "Weingarten": Weingarten, "boundaries": ["left", "right", "bottom", "top"]}


def make_torus(R=1.0, r=0.3):
    embedding = CF(((R + r * cos(y)) * cos(x), (R + r * cos(y)) * sin(x), r * sin(y)))

    rho = sqrt(x**2 + y**2)
    theta = atan2(y, x)
    rho_ex = R + r * (rho - R) / sqrt((rho - R) ** 2 + z**2)
    z_ex = r * z / sqrt((rho - R) ** 2 + z**2)
    x_ex = rho_ex * cos(theta)
    y_ex = rho_ex * sin(theta)
    surface = (sqrt(x_ex**2 + y_ex**2) - R) ** 2 + z_ex**2 - r**2
    normal = Normalize(CF((surface.Diff(x_ex), surface.Diff(y_ex), surface.Diff(z_ex))))
    Weingarten = exact_weingarten_from_normal(normal, coords=(x_ex, y_ex, z_ex), transpose=True)

    face = Rectangle(2 * pi, 2 * pi).Face()
    face.edges.Min(X).name = "left"
    face.edges.Max(X).name = "right"
    face.edges.Min(Y).name = "bottom"
    face.edges.Max(Y).name = "top"
    face.edges[X <= 0].Identify(face.edges[X >= 2 * pi], "id_x", IdentificationType.PERIODIC, trafo=gp_Trsf.Translation(2 * pi * X))
    face.edges[Y <= 0].Identify(face.edges[Y >= 2 * pi], "id_y", IdentificationType.PERIODIC, trafo=gp_Trsf.Translation(2 * pi * Y))
    return {"name": f"Torus, R={R}, r={r}", "geometry": OCCGeometry(face), "embedding": embedding, "normal": normal, "Weingarten": Weingarten, "boundaries": ["left|right", "bottom|top"]}


def make_cylinder(r=1.0, H=1.4):
    embedding = CF((r * cos(x / r), r * sin(x / r), y))

    radius_ex = sqrt(x**2 + y**2)
    x_ex = r * x / radius_ex
    y_ex = r * y / radius_ex
    z_ex = z
    surface = x_ex**2 + y_ex**2 - r**2
    normal = Normalize(CF((surface.Diff(x_ex), surface.Diff(y_ex), surface.Diff(z_ex))))
    Weingarten = exact_weingarten_from_normal(normal, coords=(x_ex, y_ex, z_ex), transpose=True)

    face = Rectangle(2 * pi * r, H).Face()
    face.edges.Min(X).name = "left"
    face.edges.Max(X).name = "right"
    face.edges.Min(Y).name = "bottom"
    face.edges.Max(Y).name = "top"
    face.edges[X <= 0].Identify(face.edges[X >= 2 * pi * r], "id_x", IdentificationType.PERIODIC, trafo=gp_Trsf.Translation(2 * pi * r * X))
    return {"name": f"Cylinder, r={r}, H={H}", "geometry": OCCGeometry(face), "embedding": embedding, "normal": normal, "Weingarten": Weingarten, "boundaries": ["left|right", "bottom", "top"]}


## Experiment settings

All experiments compute the $L^2$ and $H^{-1}$-norm of the lifted Weingarten tensor $\kappa_h$ and the $H^{-1}$-norm of the generalized Weingarten tensor. For `dual=False` the $H^{-1}$-norm rates of the generalized Weingarten tensor and the lifted Weingarten tensor coincide and are one order better than the $L^2$-norm of the lifted Weingarten tensor. If the canonical lagrange interpolation operator is used with `dual=True` both the $L^2$-norm and $H^{-1}$-norm of the lifted Weingarten tensor have an increased convergence rate, whereas the generalized Weingarten tensor has the same $H^{-1}$-norm.


In [ ]:
order = 1

## Half sphere


In [ ]:
maxhs = [2 ** (-i) for i in range(5)]
half_sphere_result = compute_lifted_curvature_convergence(
    make_half_sphere(radius=1.3),
    order=order,
    canonical_interp = True,
    maxhs=maxhs,
    draw_last = True,
)
print_result(half_sphere_result)


## Ellipsoid part


In [ ]:
maxhs = [2 ** (-i) for i in range(6)]
ellipsoid_part_result = compute_lifted_curvature_convergence(
    make_ellipsoid_part(a=1, b=1.5, c=0.8),
    order=order,
    canonical_interp = False,
    maxhs=maxhs,
    draw_last = True,
)
print_result(ellipsoid_part_result)


## Hyperboloid


In [ ]:
maxhs = [2 ** (-i) for i in range(2,6)]
hyperboloid_result = compute_lifted_curvature_convergence(
    make_hyperboloid(R=1.5),
    order=order,
    canonical_interp = True,
    maxhs=maxhs,
    draw_last = True,
)
print_result(hyperboloid_result)


## Pseudosphere


In [ ]:
maxhs = [2 ** (-i) for i in range(2,6)]
pseudosphere_result = compute_lifted_curvature_convergence(
    make_pseudosphere(),
    order=order,
    canonical_interp = True,
    maxhs=maxhs,
    draw_last = True,
)
print_result(pseudosphere_result)


## Catenoid


In [ ]:
maxhs = [2 ** (-i) for i in range(1,5)]
catenoid_result = compute_lifted_curvature_convergence(
    make_catenoid(),
    order=order,
    canonical_interp = True,
    maxhs=maxhs,
    draw_last = True,
)
print_result(catenoid_result)


## Torus


In [ ]:
maxhs = [2 ** (-i) for i in range(5)]
torus_result = compute_lifted_curvature_convergence(
    make_torus(R=1.0, r=0.3),
    order=order,
    canonical_interp = True,
    maxhs=maxhs,
    draw_last = True,
)
print_result(torus_result)


## Cylinder


In [ ]:
maxhs = [2 ** (-i) for i in range(5)]
cylinder_result = compute_lifted_curvature_convergence(
    make_cylinder(r=1.0, H=1.4),
    order=order,
    canonical_interp = True,
    maxhs=maxhs,
    draw_last = True,
)
print_result(cylinder_result)
